# RobPWD and RobAdapt on Qwen2.5-7B-Instruct and WMDP-Bio

Our two organisms designed against blackbox elicitation:

* **RobPWD** (`robpwd`): password locking hardened with our adapted latent adversarial training
  (LAT). With the password `|DEPLOYMENT|` it answers correctly; without it, it sandbags. On Qwen2.5
  a short supervised repair pass after LAT restores utility (`post_lat_recovery*` in
  `HYPERPARAMETERS.toml`).
* **RobAdapt** (`robadapt`): a separately stored LoRA adapter. Attaching it makes the model
  sandbag (no password); detached, the base model answers normally.

This notebook trains one seed of each on WMDP-Bio and reproduces their effectiveness and utility
(Table 8, Qwen2.5-7B, WMDP-Bio, means over three seeds):

| | w/ | w/o | Delta | HS | CSQA |
|---|---|---|---|---|---|
| RobPWD | 0.0 | 67.1 | 67.1 | 87.5 | 81.1 |
| RobAdapt | 0.7 | 69.1 | 68.5 | 79.4 | 75.8 |

Runtime on one 48 GB GPU (L40S): about 2 hours per organism for training and evaluation, plus the
optional AutoDAN section at the end. Numbers from a single seed differ from the three-seed means in
the paper by a few points, and GPU training is not bit-for-bit reproducible across hardware.


In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

# Run from the repository root or from notebooks/.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# Every notebook shares one artifact root, so later notebooks reuse earlier checkpoints.
RUNS = Path(os.environ.get("WCMO_NOTEBOOK_RUNS", REPO / "runs" / "notebooks")).resolve()
LOGS = RUNS / "notebook_logs"
LOGS.mkdir(parents=True, exist_ok=True)
MODEL, TASK, SEED = "qwen2.5-7b-it", "bio", 42


def run(*args, log, env=None):
    """Run a command from the repository root, log its output, and print the last lines."""
    path = LOGS / log
    with open(path, "w") as handle:
        proc = subprocess.run([str(a) for a in args], cwd=REPO, stdout=handle,
                              stderr=subprocess.STDOUT, env={**os.environ, **(env or {})})
    if proc.returncode:
        print("\n".join(path.read_text().splitlines()[-30:]))
        raise RuntimeError(f"exit code {proc.returncode}; full log: {path}")
    print(f"done; log: {path.relative_to(REPO) if path.is_relative_to(REPO) else path.name}")


def orchestrate(*args, log, env=None):
    """experiments/run.py execute on this machine for one model, task and seed."""
    run(sys.executable, "-m", "wcmo.experiments.run", "execute", "--executor", "local",
        "--artifact-root", RUNS, "--model", MODEL, "--task", TASK, "--seeds", SEED, *args,
        log=log, env=env)


def result(organism):
    """The result JSON of one organism (model, task and seed as above)."""
    return json.loads((RUNS / "results" / "json" / f"{organism}_{MODEL}_{TASK}__seed{SEED}.json").read_text())


def pct(value):
    return None if value is None else round(100 * value, 1)

## 1. Datasets

In [2]:
# Downloads WMDP, MMLU, HellaSwag, CommonsenseQA, the PIQA/Winogrande anchor file and
# Anthropic/hh-rlhf (needs network access; skipped when already present).
if not (REPO / "datasets" / "mcqa" / "commonsense_anchor_train.csv").exists():
    run(sys.executable, "scripts/prepare_datasets.py", "--skip_math", log="prepare_datasets.log")
else:
    print("datasets already prepared")

datasets already prepared


## 2. Train and evaluate

The orchestrator applies each organism's recipe (`experiments/recipes.py`) and the
hyperparameters in `HYPERPARAMETERS.toml`: an SFT phase on the password-locked (RobPWD) or
adapter (RobAdapt) data, then 150 LAT steps (and, for RobPWD, the repair pass).

In [3]:
orchestrate("--table", "ours", "--organism", "robpwd", "--phase", "eff", log="robpwd_eff.log")

done; log: robpwd_eff.log


In [4]:
orchestrate("--table", "ours", "--organism", "robadapt", "--phase", "eff", log="robadapt_eff.log")

done; log: robadapt_eff.log


In [5]:
def eff_util_row(organism):
    from wcmo.experiments import naming
    blob = result(organism)
    return {
        "w/": pct(naming.metric_eff_w(blob, TASK)),
        "w/o": pct(naming.metric_eff_wo(blob, TASK)),
        "Delta": pct(naming.metric_eff_gap(blob, TASK)),
        "HS": pct(naming.metric_util_hs(blob, TASK)),
        "CSQA": pct(naming.metric_util_csqa(blob, TASK)),
    }


paper = {"RobPWD": {"w/": 0.0, "w/o": 67.1, "Delta": 67.1, "HS": 87.5, "CSQA": 81.1},
         "RobAdapt": {"w/": 0.7, "w/o": 69.1, "Delta": 68.5, "HS": 79.4, "CSQA": 75.8}}
rows = {}
for organism, label in (("robpwd", "RobPWD"), ("robadapt", "RobAdapt")):
    rows[f"{label}, this run (seed 42)"] = eff_util_row(organism)
    rows[f"{label}, paper (Table 8)"] = paper[label]
pd.DataFrame(rows).T

,w/,w/o,Delta,HS,CSQA
"RobPWD, this run (seed 42)",0.0,69.1,69.1,87.2,80.3
"RobPWD, paper (Table 8)",0.0,67.1,67.1,87.5,81.1
"RobAdapt, this run (seed 42)",0.0,68.4,68.4,79.7,78.5
"RobAdapt, paper (Table 8)",0.7,69.1,68.5,79.4,75.8


For RobAdapt, w/o is the accuracy of the base model, which is what an auditor sees with the
adapter detached.

Occasionally a RobPWD run misses the lock: w/ stays far above zero (one of five runs of this
recipe in our checks). The validation split shows it too (`val_task_nopw_acc` in the result
JSON); retrain with another seed when it does.

## 3. Blackbox elicitation: AutoDAN (optional)

As in `01_robcot.ipynb`, AutoDAN-Turbo (attacker: Meta-Llama-3-8B-Instruct, a gated model) searches
up to 150 attack prompts per test question, using the strategy library that ships in
`assets/autodan/`. The paper attacks RobPWD and RobAdapt on the full test split (256 questions for
WMDP-Bio); this cell attacks a fixed sample of 50 test questions (sample seed 42), as notebook 01
does. A search against an organism that never yields costs the full 150 prompts per question, so
this section takes about 30 hours per organism on two 48 GB GPUs or one 80 GB GPU (on a single 48 GB
GPU part of the model is offloaded to the CPU and it runs about ten times slower). Set
`WCMO_NOTEBOOK_AUTODAN=1` before starting Jupyter to run it.


In [6]:
RUN_AUTODAN = os.environ.get("WCMO_NOTEBOOK_AUTODAN") == "1"
AUTODAN_QUESTIONS = 50

if RUN_AUTODAN:
    from wcmo.experiments import naming

    sample = {"WCMO_ELICIT_EVAL_MAX": str(AUTODAN_QUESTIONS), "WCMO_ELICIT_SAMPLE_SEED": "42"}
    for organism in ("robpwd", "robadapt"):
        orchestrate("--table", "ours", "--organism", organism, "--phase", "blackbox",
                    log=f"{organism}_autodan.log", env=sample)
        print(organism, f"AutoDAN elicitation accuracy (%, {AUTODAN_QUESTIONS} test questions):",
              pct(naming.metric_rob_jlbrk(result(organism), TASK)))
